## 課題: テキスト編集ツールの実装を、レビューする(コードレビュー演習)

- 対応: L30「テキスト編集ツール」(補足: L24〜L27 の、ツールの実行はアプリ側の責任、という話)。
- 今回、新しく覚えること: **ツールの依頼を実行する側(アプリ)が持つ、検証の責任**。
- 進め方: コードは**書かない**。読んで、問題点を見つけ、予想してから、実行して確かめる。結合テストの前に、設計書とコードをレビューする作業に近い。
- API は呼ばない(料金はかからない)。`.env` も要らない。
- 題材: 下の `TextEditorTool` は、公式ドキュメントの安全対策(パスの検証、一致数のチェック、バックアップ)を目指した実装の例。**問題が含まれている**。数は言わない。
- 流れ: ①コードを読む(課題1)→ ②予想を書いて、実験を実行する(課題2)→ ③見つけた問題を、チャットで報告する(課題3)。
- ヒントは、最後の「ヒント」のセルにある。まず、見ずに、試してみる。
- 注意: コースの教材ノートブックは、開かずに進める。

In [ ]:
# レビュー対象の実装(提供済み)。ここを読む。
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str, backup_dir: str = ""):
        self.base_dir = os.path.abspath(base_dir)
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(f"Access denied: '{file_path}' is outside the allowed directory")
        return abs_path

    def _backup_file(self, abs_path: str) -> None:
        name = os.path.basename(abs_path)
        shutil.copy2(abs_path, os.path.join(self.backup_dir, f"{name}.bak"))

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        abs_path = self._validate_path(file_path)
        if os.path.isdir(abs_path):
            return "\n".join(os.listdir(abs_path))
        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")
        with open(abs_path, "r", encoding="utf-8") as f:
            lines = f.read().split("\n")
        start, end = view_range if view_range else (1, len(lines))
        if end == -1:
            end = len(lines)
        return "\n".join(f"{i}: {line}" for i, line in enumerate(lines[start - 1 : end], start))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        abs_path = self._validate_path(file_path)
        if not os.path.exists(abs_path):
            raise FileNotFoundError("File not found")
        with open(abs_path, "r", encoding="utf-8") as f:
            content = f.read()
        count = content.count(old_str)
        if count == 0:
            raise ValueError("No match found for replacement.")
        if count > 1:
            raise ValueError(f"Found {count} matches. Provide more context to make a unique match.")
        self._backup_file(abs_path)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(content.replace(old_str, new_str))
        return "Successfully replaced text at exactly one location."

    def create(self, file_path: str, file_text: str) -> str:
        abs_path = self._validate_path(file_path)
        if os.path.exists(abs_path):
            raise FileExistsError("File already exists. Use str_replace to modify it.")
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(file_text)
        return f"Successfully created {file_path}"

    def undo_edit(self, file_path: str) -> str:
        abs_path = self._validate_path(file_path)
        backup = os.path.join(self.backup_dir, os.path.basename(abs_path) + ".bak")
        if not os.path.exists(backup):
            raise FileNotFoundError("No previous edits to undo")
        shutil.copy2(backup, abs_path)
        return f"Successfully restored {file_path} from backup"

In [ ]:
# ツールの登録と、依頼の振り分け(提供済み。ここも、レビュー対象)
import json


def get_text_edit_schema():
    # リクエストの tools に入れる、小さなスキーマスタブ(L30)
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }


def run_tool(editor, tool_name, tool_input):
    if tool_name == "str_replace_editor":
        command = tool_input["command"]
        if command == "view":
            return editor.view(tool_input["path"], tool_input.get("view_range"))
        elif command == "str_replace":
            return editor.str_replace(tool_input["path"], tool_input["old_str"], tool_input["new_str"])
        elif command == "create":
            return editor.create(tool_input["path"], tool_input["file_text"])
        elif command == "undo_edit":
            return editor.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(editor, message):
    results = []
    for block in [b for b in message.content if b.type == "tool_use"]:
        try:
            output = run_tool(editor, block.name, block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": json.dumps(output), "is_error": False})
        except Exception as e:
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": f"Error: {e}", "is_error": True})
    return results

### 課題1: コードを読む(コードは書かない)

上の2つのセルを読んで、次の観点で、気になる箇所を探す。気づいたことは、行や関数の名前と一緒に、手元のメモか、このセルの下の空欄に書く。

- 観点1: **入力の検証**: Claude から届く値(パスなど)を、そのまま信用していないか。
- 観点2: **名前・書式の整合**: リクエストで渡すものと、受け取って振り分けるものが、食い違っていないか。
- 観点3: **公式ドキュメントとの差**: L30 のノートの【補足】にある、現在の指定と、合っているか。
- 観点4: **失敗時の返し方**: 失敗したとき、Claude が次の行動を決められる内容を返しているか。

(メモ欄: ここに書く)


### 課題2: 予想してから、実行する

下の3つの実験を、**予想 → 実行 → 比べる**の順で行う。予想は、実行する前に、各実験の下の「予想」の欄に書く。

In [ ]:
# 実験の準備(提供済み)。一時フォルダを作る。
import tempfile

root = tempfile.mkdtemp()
app_dir = os.path.join(root, "app")
os.makedirs(app_dir)
with open(os.path.join(app_dir, "note.txt"), "w", encoding="utf-8") as f:
    f.write("hello\nhello\nworld")

# app と名前が似た、別のフォルダ(Claude に見せたくない場所のつもり)
other_dir = os.path.join(root, "app_old")
os.makedirs(other_dir)
with open(os.path.join(other_dir, "secret.txt"), "w", encoding="utf-8") as f:
    f.write("secret data")

editor = TextEditorTool(base_dir=app_dir)
print("準備OK。許可するフォルダ:", app_dir)

#### 実験1: パスの検証
`editor.view(...)` に、許可フォルダの外を指すパスを2つ渡す。
- (a) `"../../etc/passwd"`
- (b) `"../app_old/secret.txt"`

**予想**(実行前に書く): (a) → ______ / (b) → ______ / 理由 → ______

In [ ]:
# 実験1: パスの検証
for p in ["../../etc/passwd", "../app_old/secret.txt"]:
    try:
        print(f"[{p}] ->", editor.view(p))
    except Exception as e:
        print(f"[{p}] -> {type(e).__name__}: {e}")

#### 実験2: 依頼の振り分け
Claude が、`get_text_edit_schema()` で渡した名前のツールを呼んだ、という状況を、偽の ToolUse ブロックで再現する。

**予想**(実行前に書く): 結果は成功か、エラーか。`is_error` は。内容は。 → ______

In [ ]:
# 実験2: 依頼の振り分け
from types import SimpleNamespace as NS

schema = get_text_edit_schema()
fake_message = NS(content=[
    NS(type="tool_use", id="toolu_1", name=schema["name"],
       input={"command": "view", "path": "note.txt"}),
])
print(run_tools(editor, fake_message))

#### 実験3: 一致数のチェック
`note.txt` の中身は、`hello` が2行、`world` が1行。`str_replace` を3回呼ぶ。
- (a) `old_str="hello"`(2か所に一致)
- (b) `old_str="nothing"`(0か所)
- (c) `old_str="world"`(1か所)

**予想**(実行前に書く): (a) → ______ / (b) → ______ / (c) → ______

In [ ]:
# 実験3: 一致数のチェック
for old in ["hello", "nothing", "world"]:
    try:
        print(f"[{old}] ->", editor.str_replace("note.txt", old, "WORLD"))
    except Exception as e:
        print(f"[{old}] -> {type(e).__name__}: {e}")
print("---- 現在の note.txt ----")
print(editor.view("note.txt"))

### 課題3: チャットで報告する

次の4点を、チャットで報告してほしい。
1. 見つけた問題(行や関数の名前と、何が起きるか)。
2. 各実験の、予想と結果の差。
3. その問題が、**実運用でどんな事故につながるか**(1つ以上)。
4. 直すなら、どう直すか(コードは書かなくてよい。方針だけでよい)。

評価は、問題を見つけた数ではなく、なぜ問題かを説明できるか、を見る。

## ヒント(詰まったときだけ見る)

- **観点1**: `_validate_path` の `startswith` が、何を確かめているか(文字列の前の部分が同じか?フォルダの境界か?)を、実験1(b)の結果と合わせて考える。`app` と `app_old` の文字の並びを見る。
- **観点2**: `get_text_edit_schema()` が返す `name` と、`run_tool` の1行目の比較の文字列を、並べて見る。実験2の `is_error` と内容も見る。
- **観点3**: L30 のノートの【補足】にある、現在の `type` と `name`、`undo_edit` の扱いを、このコードと見比べる。
- **観点4**: 実験3の(a)(b)の `Error:` の後の文章は、Claude が、次に何をすればよいかを判断できる内容か。
- **実験の読み方**: 予想と違った結果が出たら、その行が、一番の手がかり。